# Q1: model comparison

Party in Thee Code. Three text-classification pipelines, using the same shared train/validation IDs as the baseline. The grids are fixed before the comparison.

In [1]:
from pathlib import Path
import json
import pandas as pd
from q1 import run_q1
from baseline import ROOT, load_data, load_split

df = load_data(ROOT / "data" / "train.csv")
train, val = load_split(df)
print("Training:", len(train), "Validation:", len(val))
print("Training labels:", train.label.value_counts().sort_index().to_dict())
print("Validation labels:", val.label.value_counts().sort_index().to_dict())

Training: 8000 Validation: 2000
Training labels: {0: 4000, 1: 4000}
Validation labels: {0: 1000, 1: 1000}


## Candidates

Word TF-IDF + logistic regression: C = 0.1, 1, 10.

Word TF-IDF + multinomial naive Bayes: alpha = 0.1, 0.5, 1.

Character TF-IDF + linear SVM: C = 0.1, 1, 10.

Each candidate fits TF-IDF on the 8,000 training texts only. AUROC uses probabilities for LR/NB and decision scores for SVM. Fit time includes the full feature pipeline and classifier. Fits use one BLAS thread.

In [2]:
tuning, comparison, selection = run_q1(ROOT / "data" / "train.csv")

word_lr, C=0.1: train=0.957064, val=0.907388, fit=4.762s
word_lr, C=1: train=0.993344, val=0.932705, fit=4.459s
word_lr, C=10: train=1.000000, val=0.942977, fit=4.720s
word_nb, alpha=0.1: train=0.990267, val=0.903779, fit=4.342s
word_nb, alpha=0.5: train=0.982729, val=0.902459, fit=4.170s
word_nb, alpha=1: train=0.976699, val=0.900577, fit=4.234s
char_svm, C=0.1: train=0.992248, val=0.964064, fit=18.024s
char_svm, C=1: train=0.999997, val=0.970704, fit=18.345s
char_svm, C=10: train=1.000000, val=0.966582, fit=19.928s

Selected: Character TF-IDF + Linear SVM
  family                                 model parameter  value  training_auroc  validation_auroc  training_time_seconds  vocabulary_size        score_type
 word_lr     Word TF-IDF + Logistic Regression         C   10.0        1.000000          0.942977               4.719706            50000       probability
 word_nb Word TF-IDF + Multinomial Naive Bayes     alpha    0.1        0.990267          0.903779               4.342300    

## Hyperparameter results

In [3]:
display(tuning[["family", "parameter", "value", "training_auroc", "validation_auroc", "training_time_seconds"]])

,family,parameter,value,training_auroc,validation_auroc,training_time_seconds
0,word_lr,C,0.1,0.957064,0.907388,4.762278
1,word_lr,C,1.0,0.993344,0.932705,4.458628
2,word_lr,C,10.0,1.000000,0.942977,4.719706
3,word_nb,alpha,0.1,0.990267,0.903779,4.342300
4,word_nb,alpha,0.5,0.982729,0.902459,4.169644
5,word_nb,alpha,1.0,0.976699,0.900577,4.233958
6,char_svm,C,0.1,0.992248,0.964064,18.024073
7,char_svm,C,1.0,0.999997,0.970704,18.344914
8,char_svm,C,10.0,1.000000,0.966582,19.928474


## Best configuration from each family

Selection is based on the same held-out validation set. These are model-selection scores, so they can be optimistic after trying multiple settings.

In [4]:
display(comparison[["model", "parameter", "value", "training_auroc", "validation_auroc", "training_time_seconds"]])
print("Selected family:", selection["family"])
print("Selected setting:", selection["parameter"], selection["value"])
print("Probability output:", selection["probability_output"])
print("Selected validation AUROC:", selection["validation_auroc"])

Selected family: char_svm
Selected setting: C 1.0
Probability output: {'method': '3-fold training-only sigmoid calibration of the full pipeline', 'ensemble': False, 'fold_seed': 42, 'extra_fit_time_seconds': 65.87750098299966}
Selected validation AUROC: 0.9707039999999999


,model,parameter,value,training_auroc,validation_auroc,training_time_seconds
0,Word TF-IDF + Logistic Regression,C,10.0,1.000000,0.942977,4.719706
1,Word TF-IDF + Multinomial Naive Bayes,alpha,0.1,0.990267,0.903779,4.342300
2,Character TF-IDF + Linear SVM,C,1.0,0.999997,0.970704,18.344914


## Saved predictions

If SVM is selected, the saved probability model uses sigmoid calibration with three folds entirely within the training partition. The full TF-IDF pipeline is refit inside each fold; the outer validation set is not used for calibration. ensemble=False then refits the base pipeline on all 8,000 training rows. The original baseline outputs stay unchanged.

In [5]:
pred = pd.read_csv(ROOT / "results" / "q1" / "selected_validation_predictions.csv", dtype={"id": str})
assert len(pred) == 2000
assert pred.id.tolist() == val.id.tolist()
assert pred.predicted_ai_probability.between(0, 1).all()
display(pred.head())
print(pred.error_type.value_counts().to_string())

error_type
correct    1828
FN           97
FP           75


,id,true_label,predicted_ai_probability,predicted_label,word_count,error_type
0,6283,0,0.046998,0,161,correct
1,2607,1,0.395829,0,179,FN
2,3980,0,0.001924,0,248,correct
3,9156,1,0.946099,1,320,correct
4,7295,0,0.006813,0,302,correct


The saved outputs are under results/q1/. The fitted probability model is saved locally under models/q1/ and is not included in the GitHub upload. No unlabeled or Kaggle test texts are used for this Q1 comparison. This choice can be updated if later team experiments improve on it.